# fair_comparison_fbcca.py

This script creates a *fairer* CCA-style baseline for comparison with TRCA.

Main differences from the original simple CCA script:
1. Apply an additional 2-40 Hz band-pass filter to the stimulus window.
2. Remove the per-trial channel mean (DC offset removal).
3. Use 5 harmonics instead of 2.
4. Use filter-bank CCA (FBCCA) instead of single-band CCA.
5. Report results at both the trial level and the run level.

Important note:
FBCCA is still an *unsupervised reference-based* method.
Even after making preprocessing and evaluation more comparable,
it is still not identical to TRCA, which is template-based and supervised.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import mne
from sklearn.cross_decomposition import CCA


## Basic settings


In [ ]:
INPUT_DIR = Path("../derived")
OUTPUT_DIR = Path("../derived")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FS = 250
N_CLASSES = 32
N_CHANNELS = 8
N_SAMPLES_STIM = 300
TRIAL_DURATION = N_SAMPLES_STIM / FS  # 1.2 seconds

# Use more harmonics than the original simple CCA.
N_HARMONICS = 5

# Filter-bank settings.
# These are intentionally simple and easy to read.
# Each tuple is (low_cut, high_cut).
FILTER_BANKS = [
    (6.0, 40.0),
    (14.0, 40.0),
    (22.0, 40.0),
]

# Standard FBCCA-style weighting.
# This matches the common pattern used in many FBCCA/TRCA examples.
FILTER_WEIGHTS = np.arange(1, len(FILTER_BANKS) + 1) ** (-1.25) + 0.25

# Output file names are separated from the original CCA outputs
# so that the old hypothesis pipeline remains untouched.
TRIAL_OUTPUT = OUTPUT_DIR / "trial_results_fbcca_fair.csv"
RUN_OUTPUT = OUTPUT_DIR / "run_summary_fbcca_fair.csv"


## Load inputs


In [ ]:
trial_metadata = pd.read_csv(INPUT_DIR / "trial_metadata.csv")
all_trials_stim = np.load(INPUT_DIR / "all_trials_stim.npy")

print("trial_metadata shape:", trial_metadata.shape)
print("all_trials_stim shape:", all_trials_stim.shape)

assert all_trials_stim.ndim == 3
assert all_trials_stim.shape[1] == N_CHANNELS
assert all_trials_stim.shape[2] == N_SAMPLES_STIM


## Define class frequencies and phases


In [ ]:
freqs = np.arange(8.0, 16.0, 1.0)
phases = np.array([0.0, 0.5 * np.pi, np.pi, 1.5 * np.pi])

class_params = []
for freq in freqs:
    for phase in phases:
        class_params.append((freq, phase))

assert len(class_params) == N_CLASSES


## Helper functions


In [ ]:
def make_reference_signal(freq, phase, n_samples, fs, n_harmonics=5):
    """
    Create sinusoidal reference signals for one class.

    Output shape is (n_samples, 2 * n_harmonics):
    [sin(1f), cos(1f), sin(2f), cos(2f), ...]
    """
    t = np.arange(n_samples) / fs
    ref_components = []

    for h in range(1, n_harmonics + 1):
        ref_components.append(np.sin(2 * np.pi * h * freq * t + h * phase))
        ref_components.append(np.cos(2 * np.pi * h * freq * t + h * phase))

    return np.column_stack(ref_components)


def cca_score(eeg_trial, ref_signal):
    """
    Compute the first canonical correlation between one EEG trial and one
    reference signal.

    eeg_trial shape: (n_channels, n_samples)
    ref_signal shape: (n_samples, n_ref_features)
    """
    X = eeg_trial.T
    Y = ref_signal

    cca = CCA(n_components=1)
    cca.fit(X, Y)
    X_c, Y_c = cca.transform(X, Y)

    r = np.corrcoef(X_c[:, 0], Y_c[:, 0])[0, 1]
    if np.isnan(r):
        return 0.0
    return float(r)


def compute_itr(accuracy, n_classes, trial_duration):
    """Compute Information Transfer Rate (bits/min)."""
    if accuracy <= 0:
        return 0.0
    if accuracy >= 1:
        return np.log2(n_classes) * 60.0 / trial_duration

    term1 = np.log2(n_classes)
    term2 = accuracy * np.log2(accuracy)
    term3 = (1 - accuracy) * np.log2((1 - accuracy) / (n_classes - 1))
    return (term1 + term2 + term3) * 60.0 / trial_duration


def bandpass_trial(eeg_trial, fs, low_cut, high_cut):
    """
    Apply band-pass filtering to one trial.

    Input shape:  (n_channels, n_samples)
    Output shape: (n_channels, n_samples)
    """
    return mne.filter.filter_data(
        data=eeg_trial,
        sfreq=fs,
        l_freq=low_cut,
        h_freq=high_cut,
        verbose=0,
        method="fir",
        fir_design="firwin",
    )


def preprocess_trial_for_fbcca(eeg_trial, fs, filter_banks):
    """
    Create filter-bank versions of one trial.

    Steps:
    1. Band-pass filter the trial in each sub-band.
    2. Remove the per-channel mean within the trial.

    Returns
    -------
    filtered_trials : list of ndarray
        Each element has shape (n_channels, n_samples).
    """
    filtered_trials = []

    for low_cut, high_cut in filter_banks:
        x = bandpass_trial(eeg_trial, fs=fs, low_cut=low_cut, high_cut=high_cut)

        # Remove the mean over time for each channel.
        # This mirrors the DC-removal step used in many TRCA pipelines.
        x = x - np.mean(x, axis=1, keepdims=True)

        filtered_trials.append(x)

    return filtered_trials


def fbcca_scores(eeg_trial, reference_signals, fs, filter_banks, filter_weights):
    """
    Compute one FBCCA score for every class.

    For each sub-band:
    - filter the EEG trial
    - compute CCA score against each class reference

    Then combine sub-band scores using weighted squared correlations.
    """
    filtered_trials = preprocess_trial_for_fbcca(
        eeg_trial=eeg_trial,
        fs=fs,
        filter_banks=filter_banks,
    )

    n_classes = len(reference_signals)
    band_scores = np.zeros((len(filter_banks), n_classes), dtype=float)

    for band_idx, x_band in enumerate(filtered_trials):
        for class_idx, ref_signal in enumerate(reference_signals):
            r = cca_score(x_band, ref_signal)
            band_scores[band_idx, class_idx] = r

    # Standard FBCCA aggregation:
    # sum_k w_k * r_k^2
    combined_scores = np.sum(filter_weights[:, None] * (band_scores ** 2), axis=0)
    return combined_scores, band_scores


## Build all reference signals once


In [ ]:
reference_signals = []
for freq, phase in class_params:
    ref_signal = make_reference_signal(
        freq=freq,
        phase=phase,
        n_samples=N_SAMPLES_STIM,
        fs=FS,
        n_harmonics=N_HARMONICS,
    )
    reference_signals.append(ref_signal)

print("Number of reference signals:", len(reference_signals))
print("One reference shape:", reference_signals[0].shape)
print("Filter bank weights:", FILTER_WEIGHTS)


## Trial-level FBCCA classification


In [ ]:
trial_result_rows = []

for i in range(len(trial_metadata)):
    eeg_trial = all_trials_stim[i]  # (8, 300)
    true_label = int(trial_metadata.loc[i, "true_label"])

    rho_all, band_scores = fbcca_scores(
        eeg_trial=eeg_trial,
        reference_signals=reference_signals,
        fs=FS,
        filter_banks=FILTER_BANKS,
        filter_weights=FILTER_WEIGHTS,
    )

    predicted_label = int(np.argmax(rho_all))
    rho_target = float(rho_all[true_label])
    rho_max = float(np.max(rho_all))

    other_mask = np.ones(len(rho_all), dtype=bool)
    other_mask[true_label] = False
    rho_best_other = float(np.max(rho_all[other_mask]))
    rho_margin = rho_target - rho_best_other
    correct = int(predicted_label == true_label)

    trial_result_rows.append({
        "participant": int(trial_metadata.loc[i, "participant"]),
        "condition": trial_metadata.loc[i, "condition"],
        "run": int(trial_metadata.loc[i, "run"]),
        "trial_repeat": int(trial_metadata.loc[i, "trial_repeat"]),
        "class_id": int(trial_metadata.loc[i, "class_id"]),
        "true_label": true_label,
        "predicted_label": predicted_label,
        "correct": correct,
        "rho_target": rho_target,
        "rho_max": rho_max,
        "rho_margin": rho_margin,
        # Helpful diagnostic columns
        "best_incorrect_score": rho_best_other,
        "band1_target_score": float(band_scores[0, true_label]),
        "band2_target_score": float(band_scores[1, true_label]),
        "band3_target_score": float(band_scores[2, true_label]),
    })

trial_results = pd.DataFrame(trial_result_rows).sort_values(
    by=["condition", "run", "trial_repeat", "class_id"]
).reset_index(drop=True)

print("trial_results shape:", trial_results.shape)
print(trial_results.head())


## Run-level summary


In [ ]:
run_summary = (
    trial_results
    .groupby(["condition", "run"], as_index=False)
    .agg(
        mean_rho_target=("rho_target", "mean"),
        mean_rho_margin=("rho_margin", "mean"),
        accuracy=("correct", "mean"),
        n_trials=("correct", "size"),
        mean_band1_target=("band1_target_score", "mean"),
        mean_band2_target=("band2_target_score", "mean"),
        mean_band3_target=("band3_target_score", "mean"),
    )
)

run_summary["ITR"] = run_summary["accuracy"].apply(
    lambda acc: compute_itr(
        accuracy=acc,
        n_classes=N_CLASSES,
        trial_duration=TRIAL_DURATION,
    )
)

run_summary = run_summary.sort_values(by=["condition", "run"]).reset_index(drop=True)

print("run_summary shape:", run_summary.shape)
print(run_summary)


## Optional grouped summaries for easier debugging


In [ ]:
condition_summary = (
    run_summary
    .groupby("condition", as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_ITR=("ITR", "mean"),
        mean_rho_target=("mean_rho_target", "mean"),
        mean_rho_margin=("mean_rho_margin", "mean"),
    )
)

print("\nCondition-level summary:")
print(condition_summary)


## Save outputs


In [ ]:
trial_results.to_csv(TRIAL_OUTPUT, index=False)
run_summary.to_csv(RUN_OUTPUT, index=False)

print("\nSaved:")
print(TRIAL_OUTPUT)
print(RUN_OUTPUT)
